# 📊 Data Exploration & Quality Analysis
## ERP Sales Analytics - Understanding the Shoebadoo E-Commerce Data

**Objective:** 
- Load and understand raw data structure
- Assess data quality (null values, duplicates, data types)
- Identify data quality issues
- Generate initial statistics and insights
- Document findings for cleaning pipeline

## 1. Setup & Imports

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, sum as spark_sum, avg, min as spark_min, max as spark_max
from pyspark.sql.types import *
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports erfolgreich!")

✅ Imports erfolgreich!


## 2. Spark Session erstellen
Da ich nicht mehr mit Databricks arbeite sondern im Docker lokal, muss ich eine Sparksession starten.

In [2]:
# Warnings zu unterdrücken:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("py4j").setLevel(logging.ERROR)


# Spark Session erstellen (ersetzt den vorhandenen 'spark' in Databricks)
spark = SparkSession.builder \
    .appName("ERP-Sales-Data-Exploration") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Spark Version anzeigen
print(f"✅ Spark Session erstellt!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/05 09:27:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/05 09:27:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Spark Session erstellt!
   Spark Version: 3.5.0
   App Name: ERP-Sales-Data-Exploration


## 3. Daten laden

**Hinweis:** Wir laden die Daten zuerst via Pandas (wie in Databricks), um INT64-Timestamp-Probleme zu umgehen.
Ich habe hier nurn DATA_PATH reingetan, um mit den Paths flexibler zu sein.

In [3]:
# Pfad-Konfiguration (Docker Container Pfad)
DATA_PATH = r"/app/data/raw"

print("📂 Lade Daten aus:", DATA_PATH)
print("-" * 80)

# Via Pandas laden (umgeht Spark Parquet INT64-Timestamp Probleme)
try:
    customers_pandas = pd.read_parquet(f"{DATA_PATH}/customers.parquet")
    print("✅ customers.parquet geladen")
    
    products_pandas = pd.read_parquet(f"{DATA_PATH}/products.parquet")
    print("✅ products.parquet geladen")
    
    returns_pandas = pd.read_parquet(f"{DATA_PATH}/returns.parquet")
    print("✅ returns.parquet geladen")
    
    sales_pandas = pd.read_parquet(f"{DATA_PATH}/sales.parquet")
    print("✅ sales.parquet geladen")
    
    print("\n🎉 Alle Dateien erfolgreich geladen!")
    
except Exception as e:
    print(f"❌ Fehler beim Laden: {e}")
    print("\n💡 Tipp: Prüfe ob die Parquet-Dateien in /app/data/raw/ liegen")

📂 Lade Daten aus: /app/data/raw
--------------------------------------------------------------------------------
✅ customers.parquet geladen
✅ products.parquet geladen
✅ returns.parquet geladen
✅ sales.parquet geladen

🎉 Alle Dateien erfolgreich geladen!


## 4. Zu Spark DataFrames konvertieren

In [4]:
# Pandas → Spark DataFrames
print("🔄 Konvertiere zu Spark DataFrames...\n")

customers_df = spark.createDataFrame(customers_pandas)
print(f"✅ customers_df: {customers_df.count():,} rows")

products_df = spark.createDataFrame(products_pandas)
print(f"✅ products_df:  {products_df.count():,} rows")

returns_df = spark.createDataFrame(returns_pandas)
print(f"✅ returns_df:   {returns_df.count():,} rows")

sales_df = spark.createDataFrame(sales_pandas)
print(f"✅ sales_df:     {sales_df.count():,} rows")

print("\n🎉 Konvertierung abgeschlossen!")

🔄 Konvertiere zu Spark DataFrames...



✅ customers_df: 8,000 rows
✅ products_df:  500 rows
✅ returns_df:   43,789 rows


25/11/05 09:27:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


✅ sales_df:     437,896 rows

🎉 Konvertierung abgeschlossen!


## 5. Schemas inspizieren

In [5]:
print("\n" + "="*80)
print(" CUSTOMERS SCHEMA")
print("="*80)
customers_df.printSchema()


 CUSTOMERS SCHEMA
root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- date_of_birth: date (nullable = true)



In [6]:
print("\n" + "="*80)
print(" PRODUCTS SCHEMA")
print("="*80)
products_df.printSchema()


 PRODUCTS SCHEMA
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
print("\n" + "="*80)
print(" SALES SCHEMA")
print("="*80)
sales_df.printSchema()


 SALES SCHEMA
root
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- channel: string (nullable = true)
 |-- sale_datetime: timestamp (nullable = true)
 |-- quantity: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)



In [8]:
print("\n" + "="*80)
print(" RETURNS SCHEMA")
print("="*80)
returns_df.printSchema()


 RETURNS SCHEMA
root
 |-- return_id: long (nullable = true)
 |-- sale_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- return_date: date (nullable = true)
 |-- return_reason: string (nullable = true)
 |-- refunded_amount: double (nullable = true)



## 6. Data Quality Report Funktion

**Verbesserte Version** meiner Quality Reports

Hier finde und zähle ich, wie in meinem alten Code, folgende Werte:
 - Null / NaN oder Empty Values
 - Duplikate von allen Spalten

Bei den Duplikaten ist zu beachten, dass es **solange es keine Duplikate bei den Primary-Keys** gibt, sind die Duplikate **nicht relevant**

In [9]:
from pyspark.sql.functions import col, isnan, trim, count as spark_count
from pyspark.sql import Window

def quality_report(df, dataset_name, primary_keys=None):
    """
    Erstellt einen ausführlichen Data Quality Report.
    
    Args:
        df: PySpark DataFrame
        dataset_name: Name des Datasets für die Ausgabe
        primary_keys: Liste der Primary Key Spalten (optional)
    
    Returns:
        dict: Zusammenfassung der Quality Metrics
    """
    print(f"\n{'='*80}")
    print(f" 📊 {dataset_name.upper()}")
    print(f"{'='*80}")
    
    # Basis Statistiken
    total = df.count()
    
    print(f"\n📈 Überblick:")
    print(f"   Total Rows:     {total:,}")
    print(f"   Columns:        {len(df.columns)}")
    
    # ========================================
    # DUPLIKATE FINDEN
    # ========================================
    
    # 1. Komplette Row-Duplikate (alle Spalten identisch)
    distinct_rows = df.distinct().count()
    full_duplicates = total - distinct_rows
    print(f"\n🔍 Komplette Row-Duplikate (alle Spalten identisch):")
    print(f"   Distinct Rows:  {distinct_rows:,}")
    print(f"   Duplicates:     {full_duplicates:,} ({full_duplicates/total*100:.2f}%)")
    
    # 2. Duplikate basierend auf Primary Keys (falls angegeben)
    if primary_keys:
        distinct_pk = df.select(primary_keys).distinct().count()
        pk_duplicates = total - distinct_pk
        print(f"\n🔑 Duplikate basierend auf Primary Keys ({', '.join(primary_keys)}):")
        print(f"   Unique Keys:    {distinct_pk:,}")
        print(f"   Duplicates:     {pk_duplicates:,} ({pk_duplicates/total*100:.2f}%)")
        
        # Zeige welche Keys dupliziert sind
        if pk_duplicates > 0:
            duplicate_keys = df.groupBy(primary_keys).count().filter("count > 1").orderBy("count", ascending=False)
            dup_key_count = duplicate_keys.count()
            print(f"   Anzahl duplizierter Keys: {dup_key_count:,}")
            print(f"\n   Top 5 am meisten duplizierte Keys:")
            duplicate_keys.limit(5).show(truncate=False)
    
    # 3. Duplikate pro einzelner Spalte
    print(f"\n📊 Unique Values pro Spalte:")
    for col_name in df.columns:
        unique_vals = df.select(col_name).distinct().count()
        dup_ratio = (total - unique_vals) / total * 100
        if dup_ratio > 50:
            status = "⚠️"  # Viele Duplikate
        elif dup_ratio > 10:
            status = "⚡"  # Einige Duplikate
        else:
            status = "✅"  # Wenig Duplikate
        print(f"   {status} {col_name:30s}: {unique_vals:>8,} unique ({dup_ratio:>5.1f}% duplicates)")
    
    # ========================================
    # NULL / MISSING VALUES
    # ========================================
    print(f"\n🔍 Missing / Problematic Values:")
    null_info = {}
    for field in df.schema.fields:
        col_name = field.name
        dt = field.dataType.simpleString()
        
        # Basis: echte NULLS
        cond = col(col_name).isNull()
        
        # Numeric: auch NaN mitzählen
        if dt in ("double", "float", "decimal", "int", "bigint", "long", "short"):
            cond = cond | isnan(col(col_name))
        
        # String: leere/whitespace Strings mitzählen
        if dt == "string":
            cond = cond | (trim(col(col_name)) == "")
        
        missing_count = df.filter(cond).count()
        null_info[col_name] = missing_count
        
        if missing_count > 0:
            percentage = missing_count / total * 100
            status = "⚠️" if percentage > 10 else "⚡"
            print(f"   {status} {col_name:30s}: {missing_count:>6,} ({percentage:>5.1f}%)")
        else:
            print(f"   ✅ {col_name:30s}: {missing_count:>6,}")
    
    # ========================================
    # DATENTYPEN
    # ========================================
    print(f"\n🏷️ Datentypen:")
    for field in df.schema.fields:
        print(f"   {field.name:30s}: {field.dataType}")
    
    # ========================================
    # SAMPLE DATA
    # ========================================
    print(f"\n📋 First 3 rows:")
    df.limit(3).show(truncate=False)
    
    # Rückgabe für spätere Analyse
    return {
        'dataset': dataset_name,
        'total_rows': total,
        'distinct_rows': distinct_rows,
        'full_duplicates': full_duplicates,
        'pk_duplicates': pk_duplicates if primary_keys else None,
        'columns': len(df.columns),
        'null_info': null_info
    }

print("✅ Quality Report Funktion definiert!")

✅ Quality Report Funktion definiert!


## 7. Quality Reports ausführen

In [12]:
# Quality Reports für alle Datasets erstellen
reports = []

reports.append(quality_report(customers_df, "CUSTOMERS"))
reports.append(quality_report(products_df, "PRODUCTS"))
reports.append(quality_report(sales_df, "SALES"))
reports.append(quality_report(returns_df, "RETURNS"))


 📊 CUSTOMERS

📈 Überblick:
   Total Rows:     8,000
   Columns:        7

🔍 Komplette Row-Duplikate (alle Spalten identisch):
   Distinct Rows:  8,000
   Duplicates:     0 (0.00%)

📊 Unique Values pro Spalte:
   ✅ customer_id                   :    8,000 unique (  0.0% duplicates)
   ⚠️ first_name                    :    1,969 unique ( 75.4% duplicates)
   ⚠️ last_name                     :      404 unique ( 95.0% duplicates)
   ✅ email                         :    7,940 unique (  0.8% duplicates)
   ⚠️ registration_date             :    1,275 unique ( 84.1% duplicates)
   ⚠️ country                       :        3 unique (100.0% duplicates)
   ⚡ date_of_birth                 :    6,533 unique ( 18.3% duplicates)

🔍 Missing / Problematic Values:
   ✅ customer_id                   :      0
   ✅ first_name                    :      0
   ✅ last_name                     :      0
   ✅ email                         :      0
   ✅ registration_date             :      0
   ✅ country          

## 8. Zusammenfassender Report
PK = Primary Key

In [17]:
print("\n" + "="*80)
print(" 📊 ZUSAMMENFASSUNG - DATA QUALITY REPORT")
print("="*80 + "\n")

for report in reports:
    print(f"Dataset: {report['dataset']}")
    print(f"  Total Rows:    {report['total_rows']:,}")
    print(f"  Distinct Rows: {report['distinct_rows']:,}")
    print(f"  PK Duplicates:    {report['full_duplicates']:,}")
    print(f"  Columns:       {report['columns']}")
    
    # Anzahl Spalten mit Nulls
    cols_with_nulls = sum(1 for count in report['null_info'].values() if count > 0)
    print(f"  Null Columns:  {cols_with_nulls}/{report['columns']}")
    print()

print("="*80)


 📊 ZUSAMMENFASSUNG - DATA QUALITY REPORT

Dataset: CUSTOMERS
  Total Rows:    8,000
  Distinct Rows: 8,000
  PK Duplicates:    0
  Columns:       7
  Null Columns:  0/7

Dataset: PRODUCTS
  Total Rows:    500
  Distinct Rows: 500
  PK Duplicates:    0
  Columns:       6
  Null Columns:  4/6

Dataset: SALES
  Total Rows:    437,896
  Distinct Rows: 437,896
  PK Duplicates:    0
  Columns:       8
  Null Columns:  1/8

Dataset: RETURNS
  Total Rows:    43,789
  Distinct Rows: 43,789
  PK Duplicates:    0
  Columns:       7
  Null Columns:  1/7



## 9. Identifizierte Probleme dokumentieren

Basierend auf den Quality Reports oben:

In [ ]:
print("\n" + "="*80)
print(" 🚨 IDENTIFIZIERTE DATENQUALITÄTSPROBLEME")
print("="*80 + "\n")

problems = []

# Duplikate
for report in reports:
    if report['duplicates'] > 0:
        problems.append(f"❌ {report['dataset']}: {report['duplicates']:,} Duplikate gefunden")

# Null Values
THRESHOLD = 0.1
for report in reports:
    for col_name, null_count in report['null_info'].items():
        if null_count > 0:
            percentage = null_count / report['total_rows'] * 100
            if percentage > THRESHOLD:  # zeig mir alle ab 0.1% an 
                problems.append(f"⚠️ {report['dataset']}.{col_name}: {percentage:.1f}% NULL")

# Ausgabe
if problems:
    for i, problem in enumerate(problems, 1):
        print(f"{i}. {problem}")
else:
    print("✅ Keine kritischen Probleme gefunden!")

print("\n" + "="*80)

## 10. Save Cleaned Data

**Optional:** Speichere die Daten für das nächste Notebook.

**Hinweis:** Spark speichert Parquett als Ordner! und die MemoryManager Warnungen sind ebenfalls normal


In [ ]:
data_cleaned_path = "/app/data/cleaned/1_data_exploration"

datasets = {
    "customers": customers_df,
    "products": products_df,
    "sales": sales_df,
    "returns": returns_df
}

for name, df in datasets.items():
    path = f"{data_cleaned_path}/{name}_clean.parquet"
    df.write.mode("overwrite").parquet(path)
    print(f"✅ {name}_clean.parquet gespeichert → {path}")

## 11. Next Steps

Based on this data exploration:

1. ✅ **Data Cleaning** → `02_data_cleaning.ipynb`
   - Remove duplicates
   - Handle missing values intelligently
   - NLP: Extract category/brand from product descriptions
   - Correct data types

2. ✅ **Data Quality Validation** → `03_data_quality_validation.ipynb`
   - Great Expectations framework setup
   - Define data contracts (JSON Schema)
   - Automated validation pipeline

3. ✅ **Dimensional Modeling** → `04_dimensional_modeling.ipynb`
   - Design star schema (Kimball methodology)
   - Build fact & dimension tables
   - Delta Lake integration

4. ✅ **Analytics Dashboard**
   - Streamlit multi-page application
   - Interactive visualizations with Plotly
   - KPI monitoring

In [ ]:
# Spark Session beenden
spark.stop()
print("\n✅ Spark Session stopped. Data Exploration complete!")

In [ ]:
import json

with open("data/clean/data_quality_summary.json", "w") as f:
    json.dump(reports, f, indent=4)
